# 🏢💀 Bankrupt-AI — przewidywanie bankructwa polskich firm
## Część 1: Dane i eksploracja

**Autor:** Wojciech Płonka (projekt indywidualny)

**Przedmiot:** Uczenie maszynowe w Python — laboratorium

---

Notebook obejmuje przygotowanie danych przed modelowaniem: wczytanie zbioru o polskich firmach,
opis zawartych w nim wskaźników finansowych, obsługę braków oraz eksplorację rozkładów i zależności.
Wytrenowane modele i ich porównanie znajdują się w osobnym notebooku (`02_modele.ipynb`).

## Konfiguracja środowiska

Analiza opiera się na standardowym zestawie bibliotek: `pandas` do operacji na tabelach, `numpy` do obliczeń,
`matplotlib` i `seaborn` do wizualizacji. Pliki źródłowe są w formacie `.arff` (format Weki), dlatego do ich
odczytu wykorzystywany jest moduł `scipy.io.arff`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import urllib.request, zipfile, io, os
from scipy.io import arff

sns.set_theme(style='whitegrid')
print('Biblioteki wczytane ✅')

## Wczytanie danych

Zbiór *Polish companies bankruptcy data* pochodzi z UCI Machine Learning Repository (zbiór nr 365).
Dane zebrał Emerging Markets Information Service na podstawie sprawozdań finansowych polskich przedsiębiorstw
produkcyjnych z lat 2000–2013.

Archiwum zawiera pięć plików (`1year.arff` … `5year.arff`) odpowiadających różnym horyzontom prognozy —
plik `Nyear` opisuje kondycję firmy na *N* lat przed momentem, w którym sprawdzano, czy zbankrutowała.
Wszystkie pięć plików łączone jest w jedną tabelę, a informacja o horyzoncie zachowana w kolumnie `forecast_year`,
co pozwala później analizować zależność trafności prognozy od odległości czasowej.

In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/365/polish+companies+bankruptcy+data.zip'

print('Pobieranie danych z UCI...')
with urllib.request.urlopen(URL) as resp:
    zip_bytes = resp.read()
print(f'Pobrano {len(zip_bytes)/1024:.0f} KB')

ramki = []
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    pliki_arff = sorted(n for n in z.namelist() if n.endswith('.arff'))
    for nazwa in pliki_arff:
        with z.open(nazwa) as f:
            dane, meta = arff.loadarff(io.TextIOWrapper(f, encoding='utf-8'))
        ramka = pd.DataFrame(dane)
        ramka['forecast_year'] = int(nazwa.replace('year.arff', '').split('/')[-1])
        ramki.append(ramka)

df = pd.concat(ramki, ignore_index=True)
print('Połączony zbiór:', df.shape)

## Wstępny przegląd

Zbiór liczy kilkadziesiąt tysięcy obserwacji opisanych 64 wskaźnikami finansowymi oraz etykietą `class`.
Etykieta w formacie `.arff` wczytuje się jako wartość bajtowa (`b'0'` / `b'1'`), więc wymaga konwersji na liczbę:
**0** oznacza firmę, która przetrwała, a **1** firmę, która zbankrutowała.

In [ ]:
df['class'] = df['class'].apply(lambda x: int(x.decode()) if isinstance(x, bytes) else int(x))
df.head()

In [ ]:
print('Liczba firm: ', df.shape[0])
print('Liczba kolumn:', df.shape[1])
print('Rozkład etykiety (0 = zdrowa, 1 = bankrut):')
print(df['class'].value_counts())

## Słownik wskaźników finansowych

Kolumny `Attr1`–`Attr64` to wskaźniki finansowe zdefiniowane w dokumentacji zbioru. Obejmują cztery główne obszary:
**rentowność** (np. zysk netto / aktywa), **zadłużenie** (zobowiązania / aktywa), **płynność** (aktywa obrotowe /
zobowiązania krótkoterminowe) oraz **sprawność operacyjną** (rotacja zapasów i należności). Pełny słownik poniżej
służy do interpretacji wyników modeli — gdy model wskaże, że dany wskaźnik silnie wpływa na ryzyko, można od razu
podać jego znaczenie ekonomiczne.

In [ ]:
opis_wskaznikow = {
    'Attr1': 'zysk netto / aktywa ogółem',
    'Attr2': 'zobowiązania ogółem / aktywa ogółem',
    'Attr3': 'kapitał obrotowy / aktywa ogółem',
    'Attr4': 'aktywa obrotowe / zobowiązania krótkoterminowe',
    'Attr5': '[(gotówka + pap. wart. + należności - zob. krótkot.) / (koszty operac. - amortyzacja)] * 365',
    'Attr6': 'zyski zatrzymane / aktywa ogółem',
    'Attr7': 'EBIT / aktywa ogółem',
    'Attr8': 'wartość księgowa kapitału własnego / zobowiązania ogółem',
    'Attr9': 'sprzedaż / aktywa ogółem',
    'Attr10': 'kapitał własny / aktywa ogółem',
    'Attr11': '(zysk brutto + poz. nadzw. + koszty finansowe) / aktywa ogółem',
    'Attr12': 'zysk brutto / zobowiązania krótkoterminowe',
    'Attr13': '(zysk brutto + amortyzacja) / sprzedaż',
    'Attr14': '(zysk brutto + odsetki) / aktywa ogółem',
    'Attr15': '(zobowiązania ogółem * 365) / (zysk brutto + amortyzacja)',
    'Attr16': '(zysk brutto + amortyzacja) / zobowiązania ogółem',
    'Attr17': 'aktywa ogółem / zobowiązania ogółem',
    'Attr18': 'zysk brutto / aktywa ogółem',
    'Attr19': 'zysk brutto / sprzedaż',
    'Attr20': '(zapasy * 365) / sprzedaż',
    'Attr21': 'sprzedaż (rok n) / sprzedaż (rok n-1)',
    'Attr22': 'zysk operacyjny / aktywa ogółem',
    'Attr23': 'zysk netto / sprzedaż',
    'Attr24': 'zysk brutto (w 3 lata) / aktywa ogółem',
    'Attr25': '(kapitał własny - kapitał akcyjny) / aktywa ogółem',
    'Attr26': '(zysk netto + amortyzacja) / zobowiązania ogółem',
    'Attr27': 'zysk operacyjny / koszty finansowe',
    'Attr28': 'kapitał obrotowy / aktywa trwałe',
    'Attr29': 'logarytm aktywów ogółem',
    'Attr30': '(zobowiązania ogółem - gotówka) / sprzedaż',
    'Attr31': '(zysk brutto + odsetki) / sprzedaż',
    'Attr32': '(zob. krótkoterminowe * 365) / koszt sprzedanych produktów',
    'Attr33': 'koszty operacyjne / zobowiązania krótkoterminowe',
    'Attr34': 'koszty operacyjne / zobowiązania ogółem',
    'Attr35': 'zysk ze sprzedaży / aktywa ogółem',
    'Attr36': 'sprzedaż ogółem / aktywa ogółem',
    'Attr37': '(aktywa obrotowe - zapasy) / zobowiązania długoterminowe',
    'Attr38': 'kapitał stały / aktywa ogółem',
    'Attr39': 'zysk ze sprzedaży / sprzedaż',
    'Attr40': '(aktywa obrotowe - zapasy - należności) / zob. krótkoterminowe',
    'Attr41': 'zobowiązania ogółem / ((zysk operac. + amortyzacja) * (12/365))',
    'Attr42': 'zysk operacyjny / sprzedaż',
    'Attr43': 'rotacja należności + rotacja zapasów (w dniach)',
    'Attr44': '(należności * 365) / sprzedaż',
    'Attr45': 'zysk netto / zapasy',
    'Attr46': '(aktywa obrotowe - zapasy) / zobowiązania krótkoterminowe',
    'Attr47': '(zapasy * 365) / koszt sprzedanych produktów',
    'Attr48': 'EBITDA (zysk operac. - amortyzacja) / aktywa ogółem',
    'Attr49': 'EBITDA / sprzedaż',
    'Attr50': 'aktywa obrotowe / zobowiązania ogółem',
    'Attr51': 'zobowiązania krótkoterminowe / aktywa ogółem',
    'Attr52': '(zob. krótkoterminowe * 365) / koszt sprzedanych produktów',
    'Attr53': 'kapitał własny / aktywa trwałe',
    'Attr54': 'kapitał stały / aktywa trwałe',
    'Attr55': 'kapitał obrotowy',
    'Attr56': '(sprzedaż - koszt sprzedanych produktów) / sprzedaż',
    'Attr57': '(aktywa obrotowe - zapasy - zob. krótkot.) / (sprzedaż - zysk brutto - amortyzacja)',
    'Attr58': 'koszty ogółem / sprzedaż ogółem',
    'Attr59': 'zobowiązania długoterminowe / kapitał własny',
    'Attr60': 'sprzedaż / zapasy',
    'Attr61': 'sprzedaż / należności',
    'Attr62': '(zob. krótkoterminowe * 365) / sprzedaż',
    'Attr63': 'sprzedaż / zobowiązania krótkoterminowe',
    'Attr64': 'sprzedaż / aktywa trwałe',
}
print(f'Słownik zawiera {len(opis_wskaznikow)} wskaźników. Przykładowe:')
for k in list(opis_wskaznikow)[:5]:
    print(f'  {k}: {opis_wskaznikow[k]}')

In [ ]:
df[['Attr1', 'Attr2', 'Attr3', 'Attr10', 'forecast_year']].describe()

## Braki danych

Dane finansowe rzadko bywają kompletne — część firm nie raportuje wszystkich pozycji. Skala braków jest tu istotna,
bo decyduje o sposobie ich obsługi. Usuwanie wierszy z brakami oznaczałoby utratę dużej części zbioru, dlatego lepszym
rozwiązaniem jest imputacja. Brakujące wartości uzupełniane są **medianą** danej kolumny — w odróżnieniu od średniej
mediana jest odporna na wartości skrajne, które we wskaźnikach finansowych występują bardzo często.

In [ ]:
braki = df.isna().sum().sort_values(ascending=False)
print('Kolumny z największą liczbą braków:')
print(braki.head(10))
print(f'\nUdział brakujących wartości w całym zbiorze: {df.isna().mean().mean()*100:.2f}%')

In [ ]:
kolumny_cech = [c for c in df.columns if c.startswith('Attr')]
df[kolumny_cech] = df[kolumny_cech].fillna(df[kolumny_cech].median())
print('Pozostałe braki:', df[kolumny_cech].isna().sum().sum())

## Eksploracja i wizualizacja

### Niezbalansowanie klas

Najważniejszą cechą zbioru jest jego silne niezbalansowanie — bankrutów jest kilkukrotnie mniej niż firm zdrowych.
Ma to bezpośredni wpływ na modelowanie: model, który zawsze przewiduje „firma przetrwa", osiągnąłby wysoką ogólną
trafność (accuracy), nie wykrywając przy tym ani jednego bankructwa. Dlatego ocena modeli musi opierać się na miarach
uwzględniających klasę mniejszościową (recall, precyzja, F1), a nie na samej trafności.

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x='class', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Liczba firm: zdrowe vs bankruci')
plt.xlabel('')
plt.ylabel('Liczba firm')
plt.xticks([0,1], ['Zdrowa (0)', 'Bankrut (1)'])
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.show()

print(f'Bankruci stanowią {df["class"].mean()*100:.1f}% wszystkich firm.')

### Bankructwa według horyzontu prognozy

Podział na pliki `1year`–`5year` pozwala sprawdzić, jak liczność firm i bankrutów rozkłada się względem odległości
czasowej od badanego momentu.

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x='forecast_year', hue='class', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Liczba firm wg horyzontu prognozy')
plt.xlabel('Lata przed badanym momentem')
plt.ylabel('Liczba firm')
plt.legend(title='', labels=['Zdrowa', 'Bankrut'])
plt.show()

### Korelacje wybranych wskaźników

Pełna macierz 64 wskaźników byłaby nieczytelna, dlatego do mapy ciepła wybrano kilkanaście reprezentatywnych miar
rentowności, zadłużenia i płynności. Interesująca jest zwłaszcza ostatnia kolumna/wiersz (`class`) — pokazuje, które
wskaźniki są najsilniej powiązane z bankructwem.

In [ ]:
wybrane = ['Attr1','Attr2','Attr3','Attr6','Attr7','Attr10','Attr16','Attr17','Attr46','class']
plt.figure(figsize=(10,8))
sns.heatmap(df[wybrane].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Mapa ciepła korelacji wybranych wskaźników')
plt.show()

### Zadłużenie a bankructwo

Wskaźnik `Attr2` (zobowiązania / aktywa) opisuje poziom zadłużenia firmy. Porównanie jego rozkładu w obu grupach
pozwala sprawdzić intuicję, że firmy bankrutujące są przeciętnie bardziej zadłużone. Skrajne 1% wartości zostało
obcięte, aby wykres pozostał czytelny mimo wartości odstających.

In [ ]:
dane = df[(df['Attr2'] > df['Attr2'].quantile(0.01)) & (df['Attr2'] < df['Attr2'].quantile(0.99))]
plt.figure(figsize=(8,4))
sns.kdeplot(data=dane, x='Attr2', hue='class', fill=True, palette=['#2ecc71', '#e74c3c'])
plt.title('Rozkład zadłużenia (Attr2 = zobowiązania / aktywa)')
plt.xlabel('zobowiązania / aktywa')
plt.show()

### Skala braków danych

Liczbowy obraz braków z poprzedniej sekcji warto pokazać graficznie. Poniższy wykres prezentuje 15 kolumn
o największej liczbie brakujących wartości (stan **przed** uzupełnieniem medianą) — od razu widać, że problem
dotyczy głównie nielicznych wskaźników, a nie całego zbioru.

In [ ]:
# korzystamy ze zmiennej `braki` policzonej PRZED uzupełnieniem
top_braki = braki.head(15)
plt.figure(figsize=(9,5))
sns.barplot(x=top_braki.values, y=top_braki.index, palette='Reds_r')
plt.title('15 kolumn z największą liczbą braków (przed uzupełnieniem)')
plt.xlabel('Liczba brakujących wartości')
plt.ylabel('Wskaźnik')
plt.tight_layout()
plt.show()

### Profil bankruta vs zdrowej firmy

To najważniejsze porównanie w całej eksploracji: zestawienie rozkładów kluczowych wskaźników w obu grupach.
Jeśli pudełka (boxploty) firm zdrowych i bankrutów wyraźnie się rozjeżdżają, oznacza to, że dany wskaźnik niesie
informację przydatną do przewidywania bankructwa. Skrajne 5% wartości obcięto dla czytelności.

In [ ]:
wskazniki = {
    'Attr2': 'Zadłużenie\n(zobow./aktywa)',
    'Attr1': 'Rentowność\n(zysk netto/aktywa)',
    'Attr6': 'Zyski zatrzymane\n/aktywa',
    'Attr10': 'Kapitał własny\n/aktywa',
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (kol, tytul) in zip(axes, wskazniki.items()):
    d = df[(df[kol] > df[kol].quantile(0.05)) & (df[kol] < df[kol].quantile(0.95))]
    sns.boxplot(x='class', y=kol, data=d, ax=ax, palette=['#2ecc71', '#e74c3c'])
    ax.set_title(tytul)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels(['Zdrowa', 'Bankrut'])
fig.suptitle('Profil finansowy: zdrowa firma vs bankrut', fontsize=14)
plt.tight_layout()
plt.show()

### Które wskaźniki najsilniej zwiastują bankructwo

Dla każdego z 64 wskaźników liczona jest korelacja z etykietą bankructwa, a następnie wybierane jest 10 o najsilniejszym
powiązaniu. Kolor czerwony oznacza, że wzrost wskaźnika zwiększa ryzyko bankructwa, a zielony — że je zmniejsza. To
zapowiedź tego, na co modele będą zwracać największą uwagę.

In [ ]:
# korelacja każdego wskaźnika z bankructwem
korelacje = df[kolumny_cech + ['class']].corr()['class'].drop('class')
top10 = korelacje[korelacje.abs().sort_values(ascending=False).head(10).index]

etykiety = [f'{nazwa} — {opis_wskaznikow.get(nazwa, "")[:32]}' for nazwa in top10.index]
kolory = ['#e74c3c' if v > 0 else '#2ecc71' for v in top10.values]

plt.figure(figsize=(10, 5))
sns.barplot(x=top10.values, y=etykiety, palette=kolory)
plt.axvline(0, color='black', lw=0.8)
plt.title('TOP 10 wskaźników najsilniej powiązanych z bankructwem')
plt.xlabel('Korelacja z bankructwem (czerwony = zwiększa ryzyko)')
plt.ylabel('')
plt.tight_layout()
plt.show()

### Rozkład rentowności w obu grupach

Histogram wskaźnika `Attr1` (zysk netto / aktywa) — podstawowej miary rentowności. Oczekujemy, że firmy zdrowe
skupiają się wokół wartości dodatnich, a bankruci są przesunięci w stronę strat. Rozkłady znormalizowano (gęstość),
aby były porównywalne mimo bardzo różnej liczebności grup.

In [ ]:
d = df[(df['Attr1'] > df['Attr1'].quantile(0.02)) & (df['Attr1'] < df['Attr1'].quantile(0.98))]
plt.figure(figsize=(9, 4))
sns.histplot(data=d, x='Attr1', hue='class', bins=50, kde=True,
             palette=['#2ecc71', '#e74c3c'], stat='density', common_norm=False)
plt.axvline(0, color='black', lw=0.8, ls='--')
plt.title('Rozkład rentowności (Attr1 = zysk netto / aktywa)')
plt.xlabel('zysk netto / aktywa')
plt.legend(title='', labels=['Bankrut', 'Zdrowa'])
plt.show()

### Tabela: średnie wartości wskaźników w obu grupach

Na koniec liczbowe podsumowanie profilu — średnia wartość wybranych wskaźników osobno dla firm zdrowych i bankrutów.
Tabela porządkuje to, co pokazały wykresy, i jest wygodna do zacytowania we wnioskach.

In [ ]:
wybrane_wsk = ['Attr1', 'Attr2', 'Attr6', 'Attr10', 'Attr16', 'Attr46']
porownanie = df.groupby('class')[wybrane_wsk].mean().T
porownanie.columns = ['Zdrowa (0)', 'Bankrut (1)']
porownanie.index = [f'{w} — {opis_wskaznikow.get(w, "")[:35]}' for w in porownanie.index]
porownanie.round(3)

## Zapis przygotowanych danych

Oczyszczony zbiór zapisywany jest do pliku CSV, dzięki czemu notebook z modelami może wczytać gotowe dane bez
powtarzania pobierania i czyszczenia.

In [ ]:
df.to_csv('dane_oczyszczone.csv', index=False)
print('Zapisano dane_oczyszczone.csv', df.shape)

---
## Podsumowanie

Zbiór realnych danych o polskich firmach został wczytany, opisany i przygotowany do modelowania. Najistotniejsze
obserwacje to silne niezbalansowanie klas oraz obecność braków danych i wartości skrajnych — wszystkie trzy cechy
wpłyną na dobór i ocenę modeli. Trenowanie i porównanie klasyfikatorów (KNN, drzewo decyzyjne, las losowy, boosting)
kontynuowane jest w notebooku `02_modele.ipynb`.